In [ ]:
import docstring_parser
from docstring_parser import parse

print("Example 1: Basic Docstring Parsing and Information Extraction")

# Google-style docstring parsing
# The opening quote starts its OWN line on purpose: recent IPython (9.x) strips doctest prompts from a triple-quoted
# literal unless nothing but a string prefix precedes the opening quote on its line. With the quote on the
# assignment line, the Example prompts (and their indent) would be stripped before the cell runs, and parse() would
# return an EMPTY Example section.
google_docstring = (
'''
Calculate the area of a rectangle.

This function takes length and width parameters and returns the area.

Args:
    length (float): The length of the rectangle
    width (float): The width of the rectangle
    units (str, optional): The units for measurement. Defaults to "cm".

Returns:
    float: The calculated area of the rectangle

Raises:
    ValueError: If length or width is negative
    TypeError: If inputs are not numeric

Example:
    >>> area = calculate_area(5.0, 3.0)
    >>> print(area)
    15.0
'''
)
assert google_docstring.count(">>>") == 2, f"IPython stripped the Example prompts from the literal: {google_docstring!r}"

# Parse the Google-style docstring
parsed = parse(google_docstring)

# Extract key information
print("=== Google Docstring Parsing ===")
print(f"Short description: {parsed.short_description}")
print(f"Long description: {parsed.long_description}")
print(f"Parameters: {len(parsed.params)} found")

for param in parsed.params:
    print(f"  - {param.arg_name} ({param.type_name}): {param.description}")
    print(f"    Optional: {param.is_optional}, Default: {param.default}")

print(f"\nReturns: {parsed.returns.type_name} - {parsed.returns.description}")
print(f"Raises: {len(parsed.raises)} exceptions")

for exc in parsed.raises:
    print(f"  - {exc.type_name}: {exc.description}")

assert parsed.short_description == "Calculate the area of a rectangle."
assert parsed.long_description == "This function takes length and width parameters and returns the area."
# "optional" in the type and "Defaults to X." in the text are both parsed; the default keeps its quotes.
assert [(p.arg_name, p.type_name, p.is_optional, p.default, p.description) for p in parsed.params] == [
    ("length", "float", False, None, "The length of the rectangle"),
    ("width", "float", False, None, "The width of the rectangle"),
    ("units", "str", True, '"cm"', 'The units for measurement. Defaults to "cm".'),
], parsed.params
assert (parsed.returns.type_name, parsed.returns.description) == ("float", "The calculated area of the rectangle")
assert [(r.type_name, r.description) for r in parsed.raises] == [
    ("ValueError", "If length or width is negative"),
    ("TypeError", "If inputs are not numeric"),
]
# The Example section keeps its doctest verbatim (it would parse as "" if IPython had stripped the prompts).
assert [e.description for e in parsed.examples] == [
    ">>> area = calculate_area(5.0, 3.0)\n>>> print(area)\n15.0"
], [e.description for e in parsed.examples]

print("Example 1 finished")

In [ ]:
from docstring_parser import parse, DocstringStyle

print("Example 2: Multi-Format Docstring Parsing (Sphinx and NumPy Styles)")

# Sphinx/reStructuredText style docstring
sphinx_docstring = '''
Process data with advanced filtering.

This function applies multiple filters to input data and returns processed results.

:param data: Input data to be processed
:type data: list
:param filters: List of filter functions to apply
:type filters: list of callable
:param strict: Whether to use strict mode processing
:type strict: bool, optional
:returns: Processed data after applying all filters
:rtype: list
:raises ProcessingError: When filter processing fails
:raises ValueError: When input data is invalid
'''

# NumPy style docstring  
numpy_docstring = '''
Compute statistical summary of dataset.

This function calculates comprehensive statistics for numerical datasets
including mean, median, standard deviation, and quartiles.

Parameters
----------
dataset : array_like
    Input numerical data for statistical analysis
axis : int, optional
    Axis along which statistics are computed (default: None)
ddof : int, optional
    Delta degrees of freedom (default: 1)

Returns
-------
dict
    Dictionary containing statistical measures:
    - mean: arithmetic mean
    - median: middle value
    - std: standard deviation
    - q1, q3: first and third quartiles

See Also
--------
numpy.mean : Compute arithmetic mean
scipy.stats : More advanced statistical functions
'''

# Parse different docstring styles
sphinx_parsed = parse(sphinx_docstring)
numpy_parsed = parse(numpy_docstring)

print("=== Sphinx Style Parsing ===")
print(f"Description: {sphinx_parsed.short_description}")
print(f"Parameters: {[p.arg_name + ':' + str(p.type_name) for p in sphinx_parsed.params]}")
print(f"Return type: {sphinx_parsed.returns.type_name if sphinx_parsed.returns else 'None'}")

print("\n=== NumPy Style Parsing ===") 
print(f"Description: {numpy_parsed.short_description}")
print(f"Parameters: {len(numpy_parsed.params)} found")
for param in numpy_parsed.params[:2]:  # Show first 2 params
    print(f"  {param.arg_name}: {param.description}")
if numpy_parsed.returns:
    print(f"Return info: {numpy_parsed.returns.description[:50]}...")

print("Example 2 finished")

In [ ]:
import inspect
from docstring_parser import parse

print("Example 3: Real Function Docstring Parsing and Metadata Extraction")

# Define a real function with comprehensive docstring
def matrix_multiply(matrix_a, matrix_b, validate=True):
    """
    Multiply two matrices using standard matrix multiplication.
    
    This function performs matrix multiplication with optional validation
    to ensure matrices have compatible dimensions.
    
    Args:
        matrix_a (list): First matrix as list of lists
        matrix_b (list): Second matrix as list of lists  
        validate (bool): Whether to validate matrix dimensions (default: True)
        
    Returns:
        list: Result matrix from multiplication
        
    Raises:
        ValueError: If matrices have incompatible dimensions
        TypeError: If inputs are not lists
        
    Example:
        >>> a = [[1, 2], [3, 4]]
        >>> b = [[5, 6], [7, 8]]
        >>> result = matrix_multiply(a, b)
        >>> print(result)
        [[19, 22], [43, 50]]
    """
    if validate:
        if not isinstance(matrix_a, list) or not isinstance(matrix_b, list):
            raise TypeError("Matrices must be lists")
        if len(matrix_a[0]) != len(matrix_b):
            raise ValueError("Incompatible matrix dimensions")
    
    # Simple matrix multiplication implementation
    result = []
    for i in range(len(matrix_a)):
        row = []
        for j in range(len(matrix_b[0])):
            sum_val = sum(matrix_a[i][k] * matrix_b[k][j] for k in range(len(matrix_b)))
            row.append(sum_val)
        result.append(row)
    return result

# Extract and parse the function's docstring
docstring = inspect.getdoc(matrix_multiply)
parsed = parse(docstring)

# Analyze the parsed docstring
print("=== Real Function Docstring Analysis ===")
print(f"Function: {matrix_multiply.__name__}")
print(f"Short description: {parsed.short_description}")
print(f"Long description: {parsed.long_description}")

print(f"\nParameters ({len(parsed.params)}):")
for param in parsed.params:
    print(f"  {param.arg_name}: {param.type_name} - {param.description}")

# Test the function to verify it matches the docstring
test_a = [[1, 2], [3, 4]]
test_b = [[5, 6], [7, 8]]
result = matrix_multiply(test_a, test_b)
print(f"\nFunction test result: {result}")
print("Docstring accurately describes function behavior!")

print("Example 3 finished")

In [ ]:
from docstring_parser import parse, Docstring, DocstringParam
from docstring_parser.common import DocstringRaises, DocstringReturns

print("Example 4: Advanced Docstring Analysis and Validation")

# Create a comprehensive docstring for analysis
# The opening quote starts its OWN line on purpose: recent IPython (9.x) strips doctest prompts from a triple-quoted
# literal unless nothing but a string prefix precedes the opening quote on its line. With the quote on the
# assignment line, the Example prompts (and their indent) would be stripped before the cell runs, and parse() would
# return an EMPTY Example section.
complex_docstring = (
'''
Advanced machine learning model trainer.

This class provides functionality for training various machine learning models
with hyperparameter optimization, cross-validation, and model persistence.

Args:
    model_type (str): Type of ML model ('svm', 'random_forest', 'neural_net')
    hyperparams (dict): Dictionary of hyperparameters for model tuning
    cross_validation (int, optional): Number of CV folds (default: 5)
    random_state (int, optional): Random seed for reproducibility (default: 42)
    verbose (bool, optional): Enable verbose logging (default: False)

Returns:
    ModelTrainer: Configured model trainer instance

Raises:
    ValueError: If model_type is not supported
    TypeError: If hyperparams is not a dictionary
    RuntimeError: If training data is insufficient

Attributes:
    model: Trained model instance
    scores: Cross-validation scores
    best_params: Optimized hyperparameters

Note:
    This trainer supports both supervised and unsupervised learning tasks.
    For best performance, ensure training data is properly preprocessed.

Example:
    >>> trainer = ModelTrainer('random_forest', {'n_estimators': 100})
    >>> trainer.fit(X_train, y_train)
    >>> predictions = trainer.predict(X_test)
'''
)
assert complex_docstring.count(">>>") == 3, f"IPython stripped the Example prompts from the literal: {complex_docstring!r}"

# Parse and analyze the complex docstring
parsed = parse(complex_docstring)

print("=== Advanced Docstring Analysis ===")
print(f"Short description: {parsed.short_description}")
print(f"Long description length: {len(parsed.long_description)} characters")

# Detailed parameter analysis
print(f"\n--- Parameter Analysis ({len(parsed.params)} params) ---")
for param in parsed.params:
    print(f"Parameter: {param.arg_name}")
    print(f"  Type: {param.type_name}")
    print(f"  Optional: {param.is_optional}")
    print(f"  Default: {param.default}")
    print(f"  Description: {param.description}")
    print()

# Exception handling analysis
print(f"--- Exception Analysis ({len(parsed.raises)} exceptions) ---")
for exc in parsed.raises:
    print(f"Exception: {exc.type_name}")
    print(f"  Condition: {exc.description}")

# Check for additional metadata sections
print("\n--- Additional Metadata ---")
if hasattr(parsed, 'meta') and parsed.meta:
    for meta_item in parsed.meta:
        print(f"{meta_item.args[0]}: {meta_item.description}")

# Validate docstring completeness
validation_score = 0
if parsed.short_description: validation_score += 1
if parsed.long_description: validation_score += 1
if parsed.params: validation_score += 1  
if parsed.returns: validation_score += 1
if parsed.raises: validation_score += 1

print(f"\nDocstring completeness score: {validation_score}/5")
assert parsed.short_description == "Advanced machine learning model trainer."
assert parsed.long_description == (
    "This class provides functionality for training various machine learning models\n"
    "with hyperparameter optimization, cross-validation, and model persistence.")
# Attributes come back in parsed.params as DocstringParam entries keyed "attribute".
assert [(p.args[0], p.arg_name, p.type_name, p.is_optional, p.description) for p in parsed.params] == [
    ("param", "model_type", "str", False, "Type of ML model ('svm', 'random_forest', 'neural_net')"),
    ("param", "hyperparams", "dict", False, "Dictionary of hyperparameters for model tuning"),
    ("param", "cross_validation", "int", True, "Number of CV folds (default: 5)"),
    ("param", "random_state", "int", True, "Random seed for reproducibility (default: 42)"),
    ("param", "verbose", "bool", True, "Enable verbose logging (default: False)"),
    ("attribute", "model", None, None, "Trained model instance"),
    ("attribute", "scores", None, None, "Cross-validation scores"),
    ("attribute", "best_params", None, None, "Optimized hyperparameters"),
], parsed.params
assert (parsed.returns.type_name, parsed.returns.description) == ("ModelTrainer", "Configured model trainer instance")
assert [(r.type_name, r.description) for r in parsed.raises] == [
    ("ValueError", "If model_type is not supported"),
    ("TypeError", "If hyperparams is not a dictionary"),
    ("RuntimeError", "If training data is insufficient"),
]
# The Example section keeps its doctest verbatim (it would parse as "" if IPython had stripped the prompts).
assert [e.description for e in parsed.examples] == [
    ">>> trainer = ModelTrainer('random_forest', {'n_estimators': 100})\n"
    ">>> trainer.fit(X_train, y_train)\n"
    ">>> predictions = trainer.predict(X_test)"
], [e.description for e in parsed.examples]
assert validation_score == 5, validation_score
print("This docstring provides comprehensive API documentation!")

print("Example 4 finished")

In [ ]:
from docstring_parser import parse
import inspect

print("Example 5: Code Generation and Documentation Tools Integration")

# Example class with various docstring styles for comparison
class DataProcessor:
    def clean_data(self, data, method='standard'):
        """
        Clean input data using specified method.
        
        Args:
            data (list): Input data to clean
            method (str): Cleaning method ('standard', 'aggressive', 'gentle')
            
        Returns:
            list: Cleaned data
        """
        if method == 'aggressive':
            return [x for x in data if x is not None and str(x).strip()]
        return [x for x in data if x is not None]
    
    def transform_data(self, data, transformers):
        """Apply multiple transformations to data.
        
        Parameters
        ----------
        data : array_like
            Input data for transformation
        transformers : list of callable
            Transformation functions to apply sequentially
            
        Returns
        -------
        transformed_data : array_like
            Data after applying all transformations
        """
        result = data
        for transformer in transformers:
            result = transformer(result)
        return result

# Dynamic docstring analysis and documentation generation
processor = DataProcessor()
methods = [getattr(processor, method) for method in dir(processor) 
           if callable(getattr(processor, method)) and not method.startswith('_')]

print("=== Dynamic Documentation Generation ===")
for method in methods:
    docstring = inspect.getdoc(method)
    if docstring:
        parsed = parse(docstring)
        
        print(f"\n**Method: {method.__name__}**")
        print(f"Description: {parsed.short_description}")
        
        # Generate parameter documentation
        if parsed.params:
            print("Parameters:")
            for param in parsed.params:
                type_info = f" ({param.type_name})" if param.type_name else ""
                print(f"  - {param.arg_name}{type_info}: {param.description}")
        
        # Generate return documentation
        if parsed.returns:
            return_type = f" ({parsed.returns.type_name})" if parsed.returns.type_name else ""
            print(f"Returns{return_type}: {parsed.returns.description}")

# Demonstrate usage with actual method calls
test_data = [1, None, 2, "", 3, None, 4]
cleaned = processor.clean_data(test_data)
print(f"\nExample usage:")
print(f"Original data: {test_data}")
print(f"Cleaned data: {cleaned}")

# Transform data example
transformers = [lambda x: [i*2 for i in x], lambda x: [i+1 for i in x]]
transformed = processor.transform_data([1, 2, 3], transformers)
print(f"Transformed [1,2,3] -> {transformed}")

print("\nDocstring parser successfully analyzed and validated method documentation!")

print("Example 5 finished")